In [ ]:
# ============================================================================
# sequencing_verification.py
#
# Purpose:
#   Verify the MILP sequencing formulation used in the OR scheduling model.
#
# Tests:
#   1. Forced ordering: Surgery 0 -> Surgery 1
#   2. Forced reverse ordering: Surgery 1 -> Surgery 0
#   3. Multi-case sequencing within one room
#   4. Different-room cases should not require sequencing
#   5. Turnover time enforcement
#   6. No-overlap validation
#
# This script mirrors the sequencing formulation currently used in
# milp_engine.ipynb.
# ============================================================================

import gurobipy as gp
from gurobipy import GRB
import itertools
import sys


# ============================================================================
# GLOBAL SETTINGS
# ============================================================================

TURNOVER = 20.0
DAY_END = 1440.0

# Use a sufficiently large M.
# This follows the same logic as the current MILP implementation:
#
# M = DAY_END + max_duration + turnover
#
# For the toy tests below, max_duration <= 300.
MAX_TEST_DURATION = 300.0

M = DAY_END + MAX_TEST_DURATION + TURNOVER


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def print_header(title):
    print("\n" + "=" * 78)
    print(title)
    print("=" * 78)


def fail(message):
    print(f"FAIL: {message}")


def passed(message):
    print(f"PASS: {message}")


# ============================================================================
# BUILD MILP MODEL
# ============================================================================
#
# This function deliberately mirrors the current sequencing formulation.
#
# Decision variables:
#
#   x[i,r]     = 1 if surgery i assigned to room r
#
#   s[i]       = start time of surgery i
#
#   c[i]       = completion time of surgery i
#
#   y[i,j,r]   = 1 if surgery i occurs before surgery j in room r
#
#
# Sequencing:
#
#   s_i + d_i + turnover
#       <= s_j
#          + M(1-y_ijr)
#          + M(2-x_ir-x_jr)
#
#   s_j + d_j + turnover
#       <= s_i
#          + M y_ijr
#          + M(2-x_ir-x_jr)
#
# ============================================================================

def build_test_model(
    durations,
    n_rooms=1,
    forced_assignment=None,
    forced_order=None,
    time_limit=30
):

    n_cases = len(durations)

    I = range(n_cases)
    R = range(n_rooms)

    model = gp.Model("SequencingVerification")

    # Silence Gurobi output.
    model.setParam("OutputFlag", 0)

    # Small toy problems should solve very quickly.
    model.setParam("TimeLimit", time_limit)
    model.setParam("MIPGap", 0.0)

    # ------------------------------------------------------------------------
    # Decision variables
    # ------------------------------------------------------------------------

    x = model.addVars(
        I,
        R,
        vtype=GRB.BINARY,
        name="x"
    )

    s = model.addVars(
        I,
        lb=0,
        ub=DAY_END,
        vtype=GRB.CONTINUOUS,
        name="start"
    )

    c = model.addVars(
        I,
        lb=0,
        vtype=GRB.CONTINUOUS,
        name="completion"
    )

    y = {}

    for i in I:
        for j in I:

            if i < j:

                for r in R:

                    y[i, j, r] = model.addVar(
                        vtype=GRB.BINARY,
                        name=f"y_{i}_{j}_{r}"
                    )

    # ------------------------------------------------------------------------
    # Assignment constraints
    # ------------------------------------------------------------------------

    for i in I:

        model.addConstr(
            gp.quicksum(
                x[i, r]
                for r in R
            ) == 1,
            name=f"assignment_{i}"
        )

    # ------------------------------------------------------------------------
    # Completion identity
    # ------------------------------------------------------------------------

    for i in I:

        model.addConstr(
            c[i] == s[i] + durations[i],
            name=f"completion_{i}"
        )

    # ------------------------------------------------------------------------
    # Linking constraints
    #
    # y[i,j,r] can only be 1 when BOTH surgeries are assigned to room r.
    # ------------------------------------------------------------------------

    for i in I:

        for j in I:

            if i < j:

                for r in R:

                    model.addConstr(
                        y[i, j, r] <= x[i, r],
                        name=f"y_link_i_{i}_{j}_{r}"
                    )

                    model.addConstr(
                        y[i, j, r] <= x[j, r],
                        name=f"y_link_j_{i}_{j}_{r}"
                    )

    # ------------------------------------------------------------------------
    # DISJUNCTIVE SEQUENCING CONSTRAINTS
    #
    # These are the exact structural constraints used in the current model.
    # ------------------------------------------------------------------------

    for i in I:

        for j in I:

            if i >= j:
                continue

            for r in R:

                # ------------------------------------------------------------
                # Case 1:
                #
                # i -> j
                #
                # If y[i,j,r] = 1 and both cases are in room r:
                #
                # s_i + d_i + turnover <= s_j
                # ------------------------------------------------------------

                model.addConstr(
                    s[i]
                    + durations[i]
                    + TURNOVER
                    <=
                    s[j]
                    + M * (1 - y[i, j, r])
                    + M * (
                        2
                        - x[i, r]
                        - x[j, r]
                    ),
                    name=f"seq_forward_{i}_{j}_{r}"
                )

                # ------------------------------------------------------------
                # Case 2:
                #
                # j -> i
                #
                # If y[i,j,r] = 0 and both cases are in room r:
                #
                # s_j + d_j + turnover <= s_i
                # ------------------------------------------------------------

                model.addConstr(
                    s[j]
                    + durations[j]
                    + TURNOVER
                    <=
                    s[i]
                    + M * y[i, j, r]
                    + M * (
                        2
                        - x[i, r]
                        - x[j, r]
                    ),
                    name=f"seq_reverse_{i}_{j}_{r}"
                )

    # ------------------------------------------------------------------------
    # Forced assignment for testing
    # ------------------------------------------------------------------------

    if forced_assignment is not None:

        for i, room in forced_assignment.items():

            model.addConstr(
                x[i, room] == 1,
                name=f"forced_assignment_{i}"
            )

    # ------------------------------------------------------------------------
    # Forced ordering for testing
    #
    # forced_order:
    #
    #     [(0,1)]
    #
    # means:
    #
    #     surgery 0 must occur before surgery 1
    #
    # Since y is only created for i < j:
    #
    #     y[0,1,r] = 1
    #
    # means 0 -> 1.
    #
    # ------------------------------------------------------------------------

    if forced_order is not None:

        for first, second in forced_order:

            if first == second:

                raise ValueError(
                    "A surgery cannot be ordered before itself."
                )

            # Normalize pair so that the dictionary key follows i < j.
            i = min(first, second)
            j = max(first, second)

            for r in R:

                if first == i and second == j:

                    # i -> j
                    model.addConstr(
                        y[i, j, r] == 1,
                        name=f"forced_forward_{i}_{j}_{r}"
                    )

                else:

                    # j -> i
                    model.addConstr(
                        y[i, j, r] == 0,
                        name=f"forced_reverse_{i}_{j}_{r}"
                    )

    # ------------------------------------------------------------------------
    # Objective
    #
    # We minimize total completion time.
    #
    # The exact objective is not the focus of this verification script.
    # It simply provides a valid optimization problem.
    # ------------------------------------------------------------------------

    model.setObjective(
        gp.quicksum(c[i] for i in I),
        GRB.MINIMIZE
    )

    return model, x, s, c, y


# ============================================================================
# EXTRACT SOLUTION
# ============================================================================

def extract_solution(
    model,
    x,
    s,
    c,
    y,
    durations,
    n_rooms
):

    n_cases = len(durations)

    assignment = {}

    for i in range(n_cases):

        assigned_room = None

        for r in range(n_rooms):

            if x[i, r].X > 0.5:

                assigned_room = r
                break

        assignment[i] = assigned_room

    start_times = {
        i: s[i].X
        for i in range(n_cases)
    }

    completion_times = {
        i: c[i].X
        for i in range(n_cases)
    }

    sequencing = []

    for (i, j, r), var in y.items():

        if var.X > 0.5:

            sequencing.append(
                {
                    "room": r,
                    "first": i,
                    "second": j
                }
            )

    return {
        "assignment": assignment,
        "start": start_times,
        "completion": completion_times,
        "sequencing": sequencing
    }


# ============================================================================
# VALIDATE SCHEDULE
# ============================================================================

def validate_schedule(
    solution,
    durations,
    turnover,
    n_rooms
):

    errors = []

    assignment = solution["assignment"]
    start = solution["start"]
    completion = solution["completion"]

    n_cases = len(durations)

    # ------------------------------------------------------------------------
    # Check assignment
    # ------------------------------------------------------------------------

    for i in range(n_cases):

        if assignment[i] is None:

            errors.append(
                f"Surgery {i} has no room assignment."
            )

    # ------------------------------------------------------------------------
    # Check completion identity
    # ------------------------------------------------------------------------

    for i in range(n_cases):

        expected_finish = (
            start[i]
            + durations[i]
        )

        if abs(
            completion[i]
            - expected_finish
        ) > 1e-5:

            errors.append(
                f"Surgery {i}: completion identity violated."
            )

    # ------------------------------------------------------------------------
    # Check every room independently
    # ------------------------------------------------------------------------

    for r in range(n_rooms):

        room_cases = [
            i
            for i in range(n_cases)
            if assignment[i] == r
        ]

        room_cases.sort(
            key=lambda i: start[i]
        )

        for pos in range(
            len(room_cases) - 1
        ):

            first = room_cases[pos]
            second = room_cases[pos + 1]

            required_start = (
                start[first]
                + durations[first]
                + turnover
            )

            actual_start = start[second]

            if actual_start + 1e-5 < required_start:

                errors.append(
                    f"Room {r}: turnover/overlap violation "
                    f"between Surgery {first} and Surgery {second}. "
                    f"Required start >= {required_start:.4f}, "
                    f"actual = {actual_start:.4f}."
                )

    return errors


# ============================================================================
# TEST 1
# ============================================================================
#
# Force:
#
#     Surgery 0 -> Surgery 1
#
# Expected:
#
#     start[1] >= start[0] + duration[0] + turnover
#
# ============================================================================

def test_forward_order():

    print_header(
        "TEST 1 — FORCED FORWARD ORDER: Surgery 0 -> Surgery 1"
    )

    durations = [
        100.0,
        200.0
    ]

    model, x, s, c, y = build_test_model(
        durations=durations,
        n_rooms=1,
        forced_assignment={
            0: 0,
            1: 0
        },
        forced_order=[
            (0, 1)
        ]
    )

    model.optimize()

    if model.Status != GRB.OPTIMAL:

        fail(
            f"Solver status = {model.Status}"
        )

        return False

    solution = extract_solution(
        model,
        x,
        s,
        c,
        y,
        durations,
        1
    )

    errors = validate_schedule(
        solution,
        durations,
        TURNOVER,
        1
    )

    if errors:

        for error in errors:
            fail(error)

        return False

    start_0 = solution["start"][0]
    start_1 = solution["start"][1]

    required = (
        start_0
        + durations[0]
        + TURNOVER
    )

    if start_1 + 1e-5 < required:

        fail(
            "Forward ordering constraint was not enforced."
        )

        return False

    passed(
        f"Forward order valid: "
        f"Surgery 0 starts at {start_0:.2f}, "
        f"Surgery 1 starts at {start_1:.2f}."
    )

    passed(
        f"Turnover verified: "
        f"Surgery 1 start >= {required:.2f}."
    )

    return True


# ============================================================================
# TEST 2
# ============================================================================
#
# Force:
#
#     Surgery 1 -> Surgery 0
#
# This is particularly important because it proves that y is NOT tied to
# the numerical case index.
# ============================================================================

def test_reverse_order():

    print_header(
        "TEST 2 — FORCED REVERSE ORDER: Surgery 1 -> Surgery 0"
    )

    durations = [
        100.0,
        200.0
    ]

    model, x, s, c, y = build_test_model(
        durations=durations,
        n_rooms=1,
        forced_assignment={
            0: 0,
            1: 0
        },
        forced_order=[
            (1, 0)
        ]
    )

    model.optimize()

    if model.Status != GRB.OPTIMAL:

        fail(
            f"Solver status = {model.Status}"
        )

        return False

    solution = extract_solution(
        model,
        x,
        s,
        c,
        y,
        durations,
        1
    )

    errors = validate_schedule(
        solution,
        durations,
        TURNOVER,
        1
    )

    if errors:

        for error in errors:
            fail(error)

        return False

    start_0 = solution["start"][0]
    start_1 = solution["start"][1]

    required = (
        start_1
        + durations[1]
        + TURNOVER
    )

    if start_0 + 1e-5 < required:

        fail(
            "Reverse ordering constraint was not enforced."
        )

        return False

    passed(
        f"Reverse order valid: "
        f"Surgery 1 starts at {start_1:.2f}, "
        f"Surgery 0 starts at {start_0:.2f}."
    )

    passed(
        f"Turnover verified: "
        f"Surgery 0 start >= {required:.2f}."
    )

    return True


# ============================================================================
# TEST 3
# ============================================================================
#
# Three surgeries in one room.
#
# Force:
#
#     0 -> 1 -> 2
#
# Expected:
#
#     s1 >= s0 + d0 + turnover
#     s2 >= s1 + d1 + turnover
#
# ============================================================================

def test_three_case_chain():

    print_header(
        "TEST 3 — THREE-CASE CHAIN: 0 -> 1 -> 2"
    )

    durations = [
        100.0,
        200.0,
        150.0
    ]

    model, x, s, c, y = build_test_model(
        durations=durations,
        n_rooms=1,
        forced_assignment={
            0: 0,
            1: 0,
            2: 0
        },
        forced_order=[
            (0, 1),
            (1, 2),
            (0, 2)
        ]
    )

    model.optimize()

    if model.Status != GRB.OPTIMAL:

        fail(
            f"Solver status = {model.Status}"
        )

        return False

    solution = extract_solution(
        model,
        x,
        s,
        c,
        y,
        durations,
        1
    )

    errors = validate_schedule(
        solution,
        durations,
        TURNOVER,
        1
    )

    if errors:

        for error in errors:
            fail(error)

        return False

    ordered = sorted(
        range(3),
        key=lambda i: solution["start"][i]
    )

    if ordered != [0, 1, 2]:

        fail(
            f"Expected order [0, 1, 2], "
            f"obtained {ordered}."
        )

        return False

    passed(
        f"Three-case chain correctly produced: "
        f"{ordered}"
    )

    for i in range(2):

        first = ordered[i]
        second = ordered[i + 1]

        gap = (
            solution["start"][second]
            -
            (
                solution["start"][first]
                + durations[first]
            )
        )

        if gap + 1e-5 < TURNOVER:

            fail(
                f"Insufficient turnover between "
                f"{first} and {second}: {gap:.4f}"
            )

            return False

        passed(
            f"Turnover {first} -> {second}: "
            f"{gap:.2f} minutes"
        )

    return True


# ============================================================================
# TEST 4
# ============================================================================
#
# Different rooms.
#
# Surgery 0 -> Room 0
# Surgery 1 -> Room 1
#
# They should be allowed to start at the same time.
#
# This tests that the sequencing constraints are correctly relaxed when
# x[i,r] + x[j,r] != 2.
# ============================================================================

def test_different_rooms():

    print_header(
        "TEST 4 — DIFFERENT ROOMS: No Cross-Room Sequencing"
    )

    durations = [
        300.0,
        300.0
    ]

    model, x, s, c, y = build_test_model(
        durations=durations,
        n_rooms=2,
        forced_assignment={
            0: 0,
            1: 1
        }
    )

    model.optimize()

    if model.Status != GRB.OPTIMAL:

        fail(
            f"Solver status = {model.Status}"
        )

        return False

    solution = extract_solution(
        model,
        x,
        s,
        c,
        y,
        durations,
        2
    )

    errors = validate_schedule(
        solution,
        durations,
        TURNOVER,
        2
    )

    if errors:

        for error in errors:
            fail(error)

        return False

    start_0 = solution["start"][0]
    start_1 = solution["start"][1]

    # Because they are in different rooms, simultaneous starts should be
    # feasible. We do not require equality, but they must both be allowed
    # without a sequencing violation.

    if abs(start_0 - start_1) < 1e-5:

        passed(
            f"Both surgeries can start simultaneously: "
            f"{start_0:.2f} min."
        )

    else:

        passed(
            f"Different-room surgeries remain feasible "
            f"without cross-room sequencing: "
            f"S0={start_0:.2f}, S1={start_1:.2f}."
        )

    return True


# ============================================================================
# TEST 5
# ============================================================================
#
# Test endogenous ordering without forcing y.
#
# Three surgeries in one room.
#
# The solver should choose SOME valid permutation rather than being forced
# to follow numerical surgery index order.
#
# ============================================================================

def test_endogenous_ordering():

    print_header(
        "TEST 5 — ENDOGENOUS ORDERING WITHOUT FORCED y"
    )

    durations = [
        100.0,
        200.0,
        150.0
    ]

    model, x, s, c, y = build_test_model(
        durations=durations,
        n_rooms=1,
        forced_assignment={
            0: 0,
            1: 0,
            2: 0
        }
    )

    model.optimize()

    if model.Status != GRB.OPTIMAL:

        fail(
            f"Solver status = {model.Status}"
        )

        return False

    solution = extract_solution(
        model,
        x,
        s,
        c,
        y,
        durations,
        1
    )

    errors = validate_schedule(
        solution,
        durations,
        TURNOVER,
        1
    )

    if errors:

        for error in errors:
            fail(error)

        return False

    ordered = sorted(
        range(3),
        key=lambda i: solution["start"][i]
    )

    # Check that all cases appear exactly once.
    if sorted(ordered) != [0, 1, 2]:

        fail(
            f"Invalid ordering returned: {ordered}"
        )

        return False

    # Check all consecutive cases.
    for pos in range(2):

        first = ordered[pos]
        second = ordered[pos + 1]

        required = (
            solution["start"][first]
            + durations[first]
            + TURNOVER
        )

        if solution["start"][second] + 1e-5 < required:

            fail(
                f"Overlap/turnover violation: "
                f"{first} -> {second}"
            )

            return False

    passed(
        f"Solver selected a valid endogenous ordering: "
        f"{ordered}"
    )

    return True


# ============================================================================
# TEST 6
# ============================================================================
#
# Explicitly inspect y variables for a 3-case room.
#
# For every pair:
#
#     exactly one direction must be represented by y
#
# when both cases are assigned to the same room.
#
# ============================================================================

def test_y_consistency():

    print_header(
        "TEST 6 — y VARIABLE CONSISTENCY"
    )

    durations = [
        120.0,
        180.0,
        90.0
    ]

    model, x, s, c, y = build_test_model(
        durations=durations,
        n_rooms=1,
        forced_assignment={
            0: 0,
            1: 0,
            2: 0
        }
    )

    model.optimize()

    if model.Status != GRB.OPTIMAL:

        fail(
            f"Solver status = {model.Status}"
        )

        return False

    solution = extract_solution(
        model,
        x,
        s,
        c,
        y,
        durations,
        1
    )

    errors = []

    # Check every pair.
    for i in range(3):

        for j in range(i + 1, 3):

            value = y[i, j, 0].X

            if value > 0.5:

                implied_first = i
                implied_second = j

            else:

                implied_first = j
                implied_second = i

            # Check that y-implied direction agrees with actual start times.
            if (
                solution["start"][implied_first]
                >
                solution["start"][implied_second]
                + 1e-5
            ):

                errors.append(
                    f"Pair ({i},{j}): "
                    f"y implies {implied_first}->{implied_second}, "
                    f"but start times imply the opposite."
                )

    if errors:

        for error in errors:
            fail(error)

        return False

    passed(
        "All y variables are consistent with actual "
        "start-time ordering."
    )

    return True


# ============================================================================
# TEST 7
# ============================================================================
#
# Full permutation test.
#
# For three cases there are 3! = 6 possible sequences.
#
# We force each possible permutation and verify that the formulation can
# represent every one.
#
# This is a particularly strong test of the endogenous sequencing logic.
# ============================================================================

def test_all_three_case_permutations():

    print_header(
        "TEST 7 — ALL 3! POSSIBLE SEQUENCING PERMUTATIONS"
    )

    durations = [
        100.0,
        150.0,
        200.0
    ]

    permutations = list(
        itertools.permutations([0, 1, 2])
    )

    all_passed = True

    for order in permutations:

        print(
            f"\nTesting forced order: {order}"
        )

        forced_pairs = [
            (order[0], order[1]),
            (order[1], order[2]),
            (order[0], order[2])
        ]

        model, x, s, c, y = build_test_model(
            durations=durations,
            n_rooms=1,
            forced_assignment={
                0: 0,
                1: 0,
                2: 0
            },
            forced_order=forced_pairs
        )

        model.optimize()

        if model.Status != GRB.OPTIMAL:

            fail(
                f"Order {order} could not be represented. "
                f"Solver status = {model.Status}"
            )

            all_passed = False
            continue

        solution = extract_solution(
            model,
            x,
            s,
            c,
            y,
            durations,
            1
        )

        actual_order = sorted(
            range(3),
            key=lambda i: solution["start"][i]
        )

        if actual_order != list(order):

            fail(
                f"Expected {order}, "
                f"obtained {actual_order}"
            )

            all_passed = False
            continue

        errors = validate_schedule(
            solution,
            durations,
            TURNOVER,
            1
        )

        if errors:

            for error in errors:
                fail(
                    f"Order {order}: {error}"
                )

            all_passed = False
            continue

        passed(
            f"Order {order} successfully represented."
        )

    return all_passed


# ============================================================================
# MAIN
# ============================================================================

def main():

    print_header(
        "MILP SEQUENCING FORMULATION VERIFICATION"
    )

    print(
        "Purpose:"
    )

    print(
        "Verify that the current room-specific binary ordering "
        "formulation correctly represents endogenous surgical sequencing."
    )

    print(
        f"\nTurnover = {TURNOVER:.1f} minutes"
    )

    print(
        f"Big-M = {M:.1f}"
    )

    print(
        "\nFormulation under test:"
    )

    print(
        "y[i,j,r] <= x[i,r]"
    )

    print(
        "y[i,j,r] <= x[j,r]"
    )

    print(
        "Forward disjunctive constraint:"
    )

    print(
        "s[i] + duration[i] + turnover"
        " <= s[j] + M(1-y) + M(2-x[i]-x[j])"
    )

    print(
        "Reverse disjunctive constraint:"
    )

    print(
        "s[j] + duration[j] + turnover"
        " <= s[i] + My + M(2-x[i]-x[j])"
    )

    print(
        "\nStarting tests..."
    )

    results = {}

    # ------------------------------------------------------------------------
    # Run tests
    # ------------------------------------------------------------------------

    results["Test 1 - Forward"] = (
        test_forward_order()
    )

    results["Test 2 - Reverse"] = (
        test_reverse_order()
    )

    results["Test 3 - Three-case chain"] = (
        test_three_case_chain()
    )

    results["Test 4 - Different rooms"] = (
        test_different_rooms()
    )

    results["Test 5 - Endogenous ordering"] = (
        test_endogenous_ordering()
    )

    results["Test 6 - y consistency"] = (
        test_y_consistency()
    )

    results["Test 7 - All permutations"] = (
        test_all_three_case_permutations()
    )

    # =========================================================================
    # FINAL REPORT
    # =========================================================================

    print_header(
        "FINAL VERIFICATION REPORT"
    )

    total = len(results)
    passed_count = sum(results.values())

    for name, result in results.items():

        if result:

            print(
                f"PASS  | {name}"
            )

        else:

            print(
                f"FAIL  | {name}"
            )

    print(
        "\n"
        + "-" * 78
    )

    print(
        f"Tests passed: {passed_count}/{total}"
    )

    print(
        f"Tests failed: {total - passed_count}/{total}"
    )

    print(
        "-" * 78
    )

    # ------------------------------------------------------------------------
    # Final conclusion
    # ------------------------------------------------------------------------

    if passed_count == total:

        print(
            "\n"
            "=============================================================="
        )

        print(
            "OVERALL RESULT: PASS"
        )

        print(
            "=============================================================="
        )

        print(
            "\nThe current MILP sequencing formulation passed all "
            "verification tests."
        )

        print(
            "\nThe tests confirm that:"
        )

        print(
            "  1. Forward ordering can be represented."
        )

        print(
            "  2. Reverse ordering can also be represented."
        )

        print(
            "  3. Multiple surgeries can be sequenced in one room."
        )

        print(
            "  4. Different rooms do not require cross-room sequencing."
        )

        print(
            "  5. Turnover time is enforced between consecutive cases."
        )

        print(
            "  6. y variables are consistent with actual start times."
        )

        print(
            "  7. All six possible three-case permutations are feasible."
        )

        print(
            "\nConclusion:"
        )

        print(
            "The sequencing formulation is computationally consistent "
            "with endogenous room-specific ordering."
        )

        print(
            "\nIMPORTANT:"
        )

        print(
            "This verifies the sequencing logic on controlled toy "
            "instances. It does not by itself prove that the complete "
            "OR optimization model is globally optimal for the full "
            "experimental dataset."
        )

        return 0

    else:

        print(
            "\n"
            "=============================================================="
        )

        print(
            "OVERALL RESULT: FAIL"
        )

        print(
            "=============================================================="
        )

        print(
            "\nAt least one sequencing verification test failed."
        )

        print(
            "Do NOT treat the current sequencing formulation as "
            "validated until the failed test has been investigated."
        )

        return 1


# ============================================================================
# SCRIPT ENTRY POINT
# ============================================================================

if __name__ == "__main__":
    main()